In [ ]:
# Cell 1 — Environment and sync.
# Set REPO_URL to this repository before running on a fresh Colab VM.
import os
from pathlib import Path

REPO_URL = os.environ.get("REPO_URL", "")
REPO_DIR = Path("/content/zalo-todo-llm-lab")

!nvidia-smi
from google.colab import drive
drive.mount("/content/drive")

if REPO_DIR.exists():
    %cd /content/zalo-todo-llm-lab
    !git pull --ff-only
elif REPO_URL:
    !git clone "$REPO_URL" /content/zalo-todo-llm-lab
    %cd /content/zalo-todo-llm-lab
else:
    raise SystemExit("Set REPO_URL to the git remote, or place the repo at /content/zalo-todo-llm-lab.")

!pip install -q -r requirements-colab.txt


In [ ]:
# Cell 2 — Zero-shot benchmark. One 4-bit model at a time, then VRAM is released.
!python -m src.eval.run_eval --models qwen2.5-7b qwen3-8b gemma2-9b gemma3-12b qwen2.5-14b --output results/benchmark_report.md
from pathlib import Path
report = Path("results/benchmark_report.md")
print(report.read_text(encoding="utf-8") if report.exists() else "No report was written.")
print("Full input/output traces:", report.with_name(report.stem + "_traces.jsonl"))


In [ ]:
# Cell 3 — Optional teacher set. The key comes from Colab Secrets, never from the notebook source.
import os
from google.colab import userdata

os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
!python -m src.dataset.synthesize --count 500


In [ ]:
# Cell 4 — QLoRA. Loss is masked to the assistant tool call. Switch --model to qwen2.5-14b if the T4 still has headroom.
!python -m src.train.train_unsloth --config configs/train_qlora.yaml --model qwen2.5-7b


In [ ]:
# Cell 5 — Score the adapter, then write Q4_K_M, IQ4_XS, and the Ollama Modelfile to Drive.
!python -m src.eval.run_eval --models qwen2.5-7b --lora-path outputs/qlora/lora --output results/benchmark_lora.md
from pathlib import Path
print(Path("results/benchmark_lora.md").read_text(encoding="utf-8"))
!python -m src.export.export_gguf --model qwen2.5-7b --lora-path outputs/qlora/lora --run-name qwen2.5-7b-qlora --output-dir /content/drive/MyDrive/zalo-todo-models/qwen2.5-7b-qlora


In [ ]:
# Cell 6 — Optional live tunnel so the Node router can call this GPU with LLM_PROVIDER=local.
import os
import time

os.environ["OLLAMA_FLASH_ATTENTION"] = "1"
os.environ["OLLAMA_KV_CACHE_TYPE"] = "q8_0"
!curl -fsSL https://ollama.com/install.sh | sh
!ollama serve > /tmp/ollama.log 2>&1 &
time.sleep(5)
!ollama create zalo-todo -f /content/drive/MyDrive/zalo-todo-models/qwen2.5-7b-qlora/Modelfile
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /tmp/cloudflared
!chmod +x /tmp/cloudflared
!/tmp/cloudflared tunnel --url http://127.0.0.1:11434 --no-autoupdate
